# Tesla Australia Customer Review NLP Analysis

**Dataset:** `tesla_synthetic_customer_reviews.csv`

This notebook analyses synthetic Tesla customer reviews from Australia to identify customer-experience themes, positive experience drivers, recurring pain points, and relationships between textual sentiment and customer satisfaction.

The marketing issue investigated is whether Tesla Australia's value proposition may increasingly depend on the wider ownership and customer experience as EV competitors such as BYD narrow gaps in price, technology, features, and range.

## 0. Setup

The notebook uses transparent, reproducible NLP methods: token frequency, n-grams, PMI, VADER sentiment, rule-based aspect detection, LDA topic modelling, and NMF topic modelling.

In [ ]:
# Run this cell once if any required libraries are missing.
%pip install pandas numpy matplotlib nltk scikit-learn scipy gensim vaderSentiment


In [ ]:
from pathlib import Path
import os
import re
import math
import unicodedata
import warnings
from collections import Counter, defaultdict

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
DATA_PATH = ROOT / "tesla_synthetic_customer_reviews.csv"
FIG_DIR = ROOT / "figures"
OUT_DIR = ROOT / "outputs"
FIG_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("default")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

RANDOM_STATE = 42

In [ ]:
import nltk
NLTK_DATA_DIR = ROOT / "nltk_data"
if NLTK_DATA_DIR.exists():
    nltk.data.path.insert(0, str(NLTK_DATA_DIR))
from nltk.util import ngrams
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def has_nltk_resource(path):
    try:
        nltk.data.find(path)
        return True
    except LookupError:
        return False

try:
    from nltk.corpus import stopwords
    STOP_WORDS_BASE = set(stopwords.words("english"))
    STOPWORD_SOURCE = "NLTK stopwords"
except LookupError:
    STOP_WORDS_BASE = set(ENGLISH_STOP_WORDS)
    STOPWORD_SOURCE = "scikit-learn fallback stopwords"

try:
    from nltk.stem import WordNetLemmatizer
    _wnl = WordNetLemmatizer()
    _wnl.lemmatize("cars", "n")
    WORDNET_AVAILABLE = True
except LookupError:
    _wnl = None
    WORDNET_AVAILABLE = False

TOKENIZER_AVAILABLE = has_nltk_resource("tokenizers/punkt") or has_nltk_resource("tokenizers/punkt_tab")
TAGGER_AVAILABLE = has_nltk_resource("taggers/averaged_perceptron_tagger_eng") or has_nltk_resource("taggers/averaged_perceptron_tagger")

print("Stopwords:", STOPWORD_SOURCE)
print("NLTK tokenizer available:", TOKENIZER_AVAILABLE)
print("NLTK POS tagger available:", TAGGER_AVAILABLE)
print("WordNet lemmatizer data available:", WORDNET_AVAILABLE)

## 1. Data Understanding

In [ ]:
df = pd.read_csv(DATA_PATH)
df["review_date"] = pd.to_datetime(df["review_date"], dayfirst=True, errors="coerce")

summary = {
    "total_reviews": len(df),
    "date_range": f"{df['review_date'].min().date()} to {df['review_date'].max().date()}",
    "vehicle_models": ", ".join(sorted(df["vehicle_model"].dropna().unique())),
    "cities": ", ".join(sorted(df["city"].dropna().unique())),
    "mean_rating": round(df["rating"].mean(), 2),
    "median_rating": float(df["rating"].median()),
    "duplicate_review_ids": int(df["review_id"].duplicated().sum()),
    "duplicate_review_texts": int(df["review_text"].duplicated().sum()),
}

display(pd.DataFrame(summary.items(), columns=["Measure", "Value"]))
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum()}))
display(df["rating"].value_counts().sort_index().rename_axis("rating").reset_index(name="count"))

df["raw_word_count"] = df["review_text"].astype(str).str.split().str.len()
display(df["raw_word_count"].describe().to_frame("review_text_word_count").round(2))
display(df.head())

## 1A. Duplicate Review Check and Cleaning Decision

The dataset has unique review IDs, but because it is synthetic, some review texts are repeated across different cities, models and ratings. This section separates three duplicate concepts: exact row duplicates, duplicate review IDs, and repeated review text. For satisfaction, city and model analysis, all review records are retained because each row has its own rating and metadata. For text/NLP interpretation, a text-deduplicated view is created as a sensitivity check so repeated synthetic wording does not dominate themes.

In [ ]:
def normalise_for_duplicate_check(text):
    text = unicodedata.normalize("NFKC", str(text)).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["review_text_norm"] = df["review_text"].apply(normalise_for_duplicate_check)

exact_row_duplicates = int(df.duplicated().sum())
duplicate_id_count = int(df["review_id"].duplicated().sum())
duplicate_text_count = int(df["review_text_norm"].duplicated().sum())
unique_text_count = int(df["review_text_norm"].nunique())

duplicate_audit = pd.DataFrame([
    {"Duplicate check": "Exact duplicate rows", "Count": exact_row_duplicates, "Cleaning decision": "No removal needed"},
    {"Duplicate check": "Duplicate review IDs", "Count": duplicate_id_count, "Cleaning decision": "No removal needed"},
    {"Duplicate check": "Repeated normalised review text", "Count": duplicate_text_count, "Cleaning decision": "Keep full dataset for rating/city/model analysis; use text-deduplicated view for NLP sensitivity"},
    {"Duplicate check": "Unique normalised review texts", "Count": unique_text_count, "Cleaning decision": "Used to assess whether repeated synthetic wording affects themes"},
])
display(duplicate_audit)

# Cleaned text-deduplicated view: keeps the first record for each repeated text only.
df_text_unique = df.sort_values("review_date").drop_duplicates(subset=["review_text_norm"], keep="first").copy()
print(f"Full review-level dataset: {len(df)} rows")
print(f"Text-deduplicated sensitivity dataset: {len(df_text_unique)} rows")
print(f"Rows removed only for text-sensitivity checks: {len(df) - len(df_text_unique)}")

repeated_text_groups = (
    df.groupby("review_text_norm")
      .agg(
          repeat_count=("review_id", "count"),
          rating_min=("rating", "min"),
          rating_max=("rating", "max"),
          cities=("city", lambda s: ", ".join(sorted(s.unique())[:6])),
          models=("vehicle_model", lambda s: ", ".join(sorted(s.unique()))),
          example_text=("review_text", "first"),
      )
      .query("repeat_count > 1")
      .sort_values("repeat_count", ascending=False)
      .head(12)
      .reset_index(drop=True)
)
display(repeated_text_groups)

**Dataset summary for the marketing report**

The dataset contains 500 synthetic Australian Tesla reviews with review ID, date, country, city, vehicle model, star rating, and review text. The analysis treats star ratings as customer satisfaction indicators and review text as evidence of experience themes. Because the data is synthetic, findings should be interpreted as analytical patterns within this dataset rather than representative evidence of real Tesla Australia customers.

## 2. Preliminary Text Inspection Before Preprocessing

In [ ]:
BASIC_STOP = set(STOP_WORDS_BASE)

def simple_tokens(text):
    text = unicodedata.normalize("NFKC", str(text).lower())
    return re.findall(r"[a-z][a-z']+", text)

raw_tokens = [tok for text in df["review_text"] for tok in simple_tokens(text)]
raw_tokens_no_stop = [t for t in raw_tokens if t not in BASIC_STOP and len(t) > 1]

def top_ngrams(token_lists, n=2, top=20):
    counts = Counter()
    for toks in token_lists:
        counts.update([" ".join(g) for g in ngrams(toks, n)])
    return pd.DataFrame(counts.most_common(top), columns=[f"{n}-gram", "count"])

raw_token_lists = [[t for t in simple_tokens(text) if t not in BASIC_STOP and len(t) > 1] for text in df["review_text"]]
raw_words = pd.DataFrame(Counter(raw_tokens_no_stop).most_common(20), columns=["word", "count"])
raw_bigrams = top_ngrams(raw_token_lists, 2, 20)
raw_trigrams = top_ngrams(raw_token_lists, 3, 20)

display(raw_words)
display(raw_bigrams)
display(raw_trigrams)

Preliminary frequencies are used only to understand vocabulary and noise. They are not final insights because stopwords, inflections, and repeated generic wording can distort simple counts.

## 3. Text Preprocessing

In [ ]:
stop_words = set(STOP_WORDS_BASE)

# Keep customer-experience words even if they appear often.
preserve_terms = {
    "service", "battery", "charging", "range", "software", "price",
    "vehicle", "model", "quality", "delivery", "charge", "charger"
}
stop_words = stop_words - preserve_terms

def nltk_to_wordnet_pos(tag):
    if tag.startswith("J"):
        return "a"
    if tag.startswith("V"):
        return "v"
    if tag.startswith("N"):
        return "n"
    if tag.startswith("R"):
        return "r"
    return "n"

def fallback_tokenize(text):
    return re.findall(r"[a-z][a-z']+", text)

def tokenize_text(text):
    if TOKENIZER_AVAILABLE:
        from nltk import word_tokenize
        return word_tokenize(text)
    return fallback_tokenize(text)

def tag_tokens(tokens):
    if TAGGER_AVAILABLE:
        from nltk import pos_tag
        return pos_tag(tokens)
    # Conservative fallback: treat tokens as nouns so downstream code is stable.
    return [(token, "NN") for token in tokens]

def lemmatize_token(token, pos):
    if WORDNET_AVAILABLE and _wnl is not None:
        return _wnl.lemmatize(token, pos)
    return token

def preprocess_review(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)
    text = text.lower()
    text = re.sub(r"[^a-z\s']", " ", text)
    text = re.sub(r"\b\d+\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = tokenize_text(text)
    tagged = tag_tokens(tokens)
    clean_tokens = []
    for token, tag in tagged:
        token = token.strip("'")
        if not token or token in stop_words or len(token) < 2:
            continue
        lemma = lemmatize_token(token, nltk_to_wordnet_pos(tag))
        if lemma and lemma not in stop_words:
            clean_tokens.append(lemma)
    return clean_tokens

df["tokens"] = df["review_text"].apply(preprocess_review)
df["clean_text"] = df["tokens"].apply(lambda toks: " ".join(toks))
df["word_count"] = df["tokens"].apply(len)

display(df[["review_text", "clean_text", "tokens", "word_count"]].head())
display(df["word_count"].describe().to_frame("cleaned_word_count").round(2))

## 4. Cleaned Frequency Analysis

In [ ]:
all_tokens = [tok for toks in df["tokens"] for tok in toks]
token_counts = Counter(all_tokens)
total_tokens = sum(token_counts.values())
top_words = pd.DataFrame(token_counts.most_common(20), columns=["term", "count"])
top_words["contribution_pct"] = (top_words["count"] / total_tokens * 100).round(2)
display(top_words)

fig, ax = plt.subplots(figsize=(9, 6))
plot_df = top_words.sort_values("count")
ax.barh(plot_df["term"], plot_df["count"], color="#3264a8")
ax.set_title("Top 20 Meaningful Terms in Cleaned Tesla Reviews")
ax.set_xlabel("Frequency")
ax.set_ylabel("Term")
plt.tight_layout()
plt.savefig(FIG_DIR / "top_meaningful_terms.png", dpi=180)
plt.show()

The cleaned frequency analysis helps identify the customer-experience areas that dominate review language. Terms that repeatedly appear across reviews are treated as candidate themes and are checked against n-grams, PMI, aspect analysis, and topic models before being promoted to final insights.

## 5. N-Gram Analysis

In [ ]:
cleaned_bigrams = top_ngrams(df["tokens"], 2, 30)
cleaned_trigrams = top_ngrams(df["tokens"], 3, 30)
cleaned_bigrams["pct_all_bigrams"] = (cleaned_bigrams["count"] / max(1, cleaned_bigrams["count"].sum()) * 100).round(2)
cleaned_trigrams["pct_all_trigrams"] = (cleaned_trigrams["count"] / max(1, cleaned_trigrams["count"].sum()) * 100).round(2)
display(cleaned_bigrams)
display(cleaned_trigrams)

## 6. Word Association Using PMI

In [ ]:
def calculate_bigram_pmi(token_lists, min_freq=4):
    unigram_counts = Counter()
    bigram_counts = Counter()
    total_unigrams = 0
    total_bigrams = 0
    for toks in token_lists:
        unigram_counts.update(toks)
        bigram_counts.update(zip(toks, toks[1:]))
        total_unigrams += len(toks)
        total_bigrams += max(0, len(toks) - 1)

    rows = []
    for (w1, w2), freq in bigram_counts.items():
        if freq < min_freq:
            continue
        p_xy = freq / total_bigrams
        p_x = unigram_counts[w1] / total_unigrams
        p_y = unigram_counts[w2] / total_unigrams
        pmi = math.log2(p_xy / (p_x * p_y))
        rows.append({"bigram": f"{w1} {w2}", "frequency": freq, "pmi": pmi})
    return pd.DataFrame(rows).sort_values(["pmi", "frequency"], ascending=[False, False]).reset_index(drop=True)

pmi_bigrams = calculate_bigram_pmi(df["tokens"], min_freq=4)
display(pmi_bigrams.head(25))
display(pmi_bigrams.sort_values("frequency", ascending=False).head(25))

## 7. Phrase Detection

In [ ]:
try:
    from gensim.models.phrases import Phrases, Phraser
    phrase_status = "Gensim phrase detection available."
    bigram_model = Phrases(df["tokens"], min_count=4, threshold=8)
    bigram_phraser = Phraser(bigram_model)
    bigram_tokens = [bigram_phraser[toks] for toks in df["tokens"]]
    trigram_model = Phrases(bigram_tokens, min_count=3, threshold=8)
    trigram_phraser = Phraser(trigram_model)
    df["phrased_tokens"] = [trigram_phraser[bigram_phraser[toks]] for toks in df["tokens"]]
except Exception as exc:
    phrase_status = f"Gensim phrase detection skipped: {exc}"
    df["phrased_tokens"] = df["tokens"]

phrase_counts = Counter(tok for toks in df["phrased_tokens"] for tok in toks if "_" in tok)
print(phrase_status)
display(pd.DataFrame(phrase_counts.most_common(25), columns=["phrase", "count"]))

## 8. Overall Customer Satisfaction

In [ ]:
rating_counts = df["rating"].value_counts().sort_index().rename_axis("rating").reset_index(name="count")
rating_counts["pct"] = (rating_counts["count"] / len(df) * 100).round(1)
satisfaction_summary = pd.DataFrame({
    "metric": [
        "Average rating",
        "Median rating",
        "1-2 star reviews",
        "3-star reviews",
        "4-5 star reviews",
    ],
    "value": [
        round(df["rating"].mean(), 2),
        df["rating"].median(),
        f"{(df['rating'].le(2).mean()*100):.1f}%",
        f"{(df['rating'].eq(3).mean()*100):.1f}%",
        f"{(df['rating'].ge(4).mean()*100):.1f}%",
    ],
})
display(rating_counts)
display(satisfaction_summary)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(rating_counts["rating"].astype(str), rating_counts["pct"], color="#3b7a57")
ax.set_title("Star Rating Distribution")
ax.set_xlabel("Star rating")
ax.set_ylabel("Reviews (%)")
for i, row in rating_counts.iterrows():
    ax.text(i, row["pct"] + 0.7, f"{row['pct']}%", ha="center")
plt.tight_layout()
plt.savefig(FIG_DIR / "rating_distribution.png", dpi=180)
plt.show()

## 9. Sentiment Analysis

In [ ]:
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer as VaderSentimentAnalyzer
    sia = VaderSentimentAnalyzer()
    SENTIMENT_METHOD = "vaderSentiment package"
except Exception:
    from nltk.sentiment import SentimentIntensityAnalyzer
    sia = SentimentIntensityAnalyzer()
    SENTIMENT_METHOD = "NLTK VADER"

print("Sentiment method:", SENTIMENT_METHOD)
df["sentiment_score"] = df["review_text"].apply(lambda x: sia.polarity_scores(str(x))["compound"])

def sentiment_label(score):
    if score >= 0.05:
        return "Positive"
    if score <= -0.05:
        return "Negative"
    return "Neutral"

df["sentiment_label"] = df["sentiment_score"].apply(sentiment_label)
sentiment_summary = df["sentiment_label"].value_counts().reindex(["Positive", "Neutral", "Negative"], fill_value=0).rename_axis("sentiment").reset_index(name="count")
sentiment_summary["pct"] = (sentiment_summary["count"] / len(df) * 100).round(1)
display(sentiment_summary)
print("Average sentiment score:", round(df["sentiment_score"].mean(), 3))

fig, ax = plt.subplots(figsize=(7, 4.5))
colors = {"Positive": "#3b7a57", "Neutral": "#8a8f98", "Negative": "#b54a4a"}
ax.bar(sentiment_summary["sentiment"], sentiment_summary["pct"], color=[colors[s] for s in sentiment_summary["sentiment"]])
ax.set_title("VADER Sentiment Distribution")
ax.set_xlabel("Sentiment label")
ax.set_ylabel("Reviews (%)")
for i, row in sentiment_summary.iterrows():
    ax.text(i, row["pct"] + 0.7, f"{row['pct']}%", ha="center")
plt.tight_layout()
plt.savefig(FIG_DIR / "sentiment_distribution.png", dpi=180)
plt.show()

## 10. Rating and Sentiment Relationship

In [ ]:
def rating_group(rating):
    if rating <= 2:
        return "Negative"
    if rating == 3:
        return "Neutral"
    return "Positive"

df["rating_group"] = df["rating"].apply(rating_group)
sentiment_crosstab = pd.crosstab(df["rating_group"], df["sentiment_label"]).reindex(index=["Negative", "Neutral", "Positive"], columns=["Positive", "Neutral", "Negative"], fill_value=0)
display(sentiment_crosstab)

df["rating_text_agree"] = df["rating_group"].eq(df["sentiment_label"])
agreement_rate = df["rating_text_agree"].mean()
print(f"Overall agreement between rating group and text sentiment: {agreement_rate:.1%}")

mismatches = df.loc[~df["rating_text_agree"], ["review_id", "rating", "rating_group", "sentiment_label", "sentiment_score", "vehicle_model", "city", "review_text"]]
display(mismatches.head(12))

## 11. Discover and Analyse Customer Experience Themes

In [ ]:
theme_keywords = {
    "Driving performance": {"drive", "driving", "performance", "acceleration", "smooth", "handling", "lane", "cruise"},
    "Battery, range and charging": {"battery", "range", "charge", "charging", "charger", "home", "supercharger"},
    "Technology and software": {"technology", "software", "update", "app", "screen", "system", "autopilot", "autonomous", "adaptive"},
    "Build quality and interior": {"build", "quality", "cabin", "interior", "seat", "panel", "noise", "luxurious"},
    "Value and price": {"value", "price", "cost", "expensive", "saving", "money"},
    "Delivery and sales": {"delivery", "deliver", "sales", "pickup", "handover", "buying", "purchase"},
    "Service and repair": {"service", "repair", "appointment", "technician", "booking", "wait", "centre", "center", "fixed", "staff", "support"},
    "Ownership trust and expectations": {"ownership", "trust", "expectation", "issue", "problem", "experience", "reliable"},
}

for theme, keywords in theme_keywords.items():
    df[f"theme__{theme}"] = df["tokens"].apply(lambda toks, keys=keywords: bool(set(toks) & keys))

def summarise_theme(theme):
    mask = df[f"theme__{theme}"]
    subset = df[mask]
    sentiment_mix = (
        subset["sentiment_label"]
        .value_counts(normalize=True)
        .reindex(["Positive", "Neutral", "Negative"], fill_value=0)
        * 100
    )
    return {
        "Theme": theme,
        "Keywords / phrases": ", ".join(sorted(theme_keywords[theme])),
        "Mentions": int(mask.sum()),
        "% Reviews": round(mask.mean() * 100, 1),
        "Avg Rating": round(subset["rating"].mean(), 2),
        "Positive %": round(sentiment_mix["Positive"], 1),
        "Neutral %": round(sentiment_mix["Neutral"], 1),
        "Negative %": round(sentiment_mix["Negative"], 1),
        "Avg Sentiment": round(subset["sentiment_score"].mean(), 3),
    }

theme_summary = (
    pd.DataFrame(summarise_theme(theme) for theme in theme_keywords)
    .sort_values(["Mentions", "Negative %"], ascending=[False, False])
    .reset_index(drop=True)
)
display(theme_summary)
theme_summary.to_csv(OUT_DIR / "theme_summary.csv", index=False)

def label_horizontal_bars(ax, suffix="", decimals=1):
    for bar in ax.patches:
        value = bar.get_width()
        label = f"{value:.{decimals}f}{suffix}"
        ax.text(value + ax.get_xlim()[1] * 0.01, bar.get_y() + bar.get_height() / 2, label, va="center", fontsize=9)

fig, ax = plt.subplots(figsize=(10, 6))
plot = theme_summary.sort_values("Mentions")
ax.barh(plot["Theme"], plot["Mentions"], color="#3264a8")
ax.set_title("Customer Experience Theme Frequency")
ax.set_xlabel("Reviews mentioning theme")
ax.set_ylabel("Theme")
plt.tight_layout()
plt.savefig(FIG_DIR / "theme_frequency.png", dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
plot = theme_summary.sort_values("Negative %", ascending=True)
left = np.zeros(len(plot))
for label, color in [("Positive %", "#3b7a57"), ("Neutral %", "#8a8f98"), ("Negative %", "#b54a4a")]:
    ax.barh(plot["Theme"], plot[label], left=left, label=label.replace(" %", ""), color=color)
    left += plot[label].to_numpy()
ax.set_title("Sentiment Mix by Customer Experience Theme")
ax.set_xlabel("Reviews mentioning theme (%)")
ax.set_ylabel("Theme")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(FIG_DIR / "sentiment_by_theme.png", dpi=180)
plt.show()

major_theme_metrics = theme_summary.head(4).copy()
y_pos = np.arange(len(major_theme_metrics))

fig, axes = plt.subplots(1, 3, figsize=(15, 5.5), sharey=True)
comparisons = [
    ("% Reviews", "Theme frequency (% of reviews)", "#3264a8", "%", 1, 50),
    ("Negative %", "Negative VADER sentiment (%)", "#b54a4a", "%", 1, 35),
    ("Avg Rating", "Average star rating", "#3b7a57", "", 2, 5),
]

for ax, (column, title, color, suffix, decimals, x_max) in zip(axes, comparisons):
    ax.barh(y_pos, major_theme_metrics[column], color=color)
    ax.set_title(title, fontsize=11)
    ax.set_xlim(0, x_max)
    ax.set_yticks(y_pos, major_theme_metrics["Theme"])
    ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.25)
    label_horizontal_bars(ax, suffix=suffix, decimals=decimals)

axes[0].set_ylabel("Customer-experience theme")
fig.suptitle(
    "Comparison of Major Tesla Customer-Experience Themes by Frequency, Negative Sentiment and Rating",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.savefig(FIG_DIR / "major_theme_frequency_sentiment_rating_comparison.png", dpi=180)
plt.show()


## 12. 2-Star Reviews With Positive Text Sentiment

A 2-star rating signals dissatisfaction, but VADER can still label the text positive when the review contains positive words such as *like*, *good*, *helpful*, *advanced*, *smooth* or *excellent*. These cases are analytically useful because they often show mixed experiences: customers may value the vehicle but still give a low rating because of service, charging, software, range anxiety or trust problems.

In [ ]:
two_star_positive = df[(df["rating"].eq(2)) & (df["sentiment_label"].eq("Positive"))].copy()
two_star_all = df[df["rating"].eq(2)].copy()

print(f"2-star reviews with positive text sentiment: {len(two_star_positive)} of {len(two_star_all)} 2-star reviews ({len(two_star_positive) / max(1, len(two_star_all)):.1%})")
print(f"Share of all reviews: {len(two_star_positive) / len(df):.1%}")
print(f"Average sentiment score among these mismatches: {two_star_positive['sentiment_score'].mean():.3f}")

# Compare themes in all 2-star reviews vs the 2-star-positive mismatch subset.
theme_compare_rows = []
for theme in theme_keywords:
    col = f"theme__{theme}"
    theme_compare_rows.append({
        "Theme": theme,
        "2-star positive sentiment mentions": int(two_star_positive[col].sum()),
        "2-star positive sentiment %": round(two_star_positive[col].mean() * 100, 1),
        "All 2-star mentions": int(two_star_all[col].sum()),
        "All 2-star %": round(two_star_all[col].mean() * 100, 1),
        "Difference pp": round((two_star_positive[col].mean() - two_star_all[col].mean()) * 100, 1),
    })
two_star_theme_compare = pd.DataFrame(theme_compare_rows).sort_values("2-star positive sentiment %", ascending=False)
display(two_star_theme_compare)

positive_cue_terms = {
    "like", "good", "great", "helpful", "smooth", "excellent", "advanced", "amazing", "useful", "comfortable", "appreciate", "easy", "seamless", "polite", "reliable"
}
negative_issue_terms = {
    "but", "however", "although", "not", "inconsistent", "stressful", "frustrating", "frustration", "problem", "issue", "wait", "delayed", "poor", "anxiety", "less", "waste", "nightmare", "oversells", "unrealistic", "expected", "trust"
}

def count_terms(tokens, terms):
    return sum(1 for token in tokens if token in terms)

two_star_positive["positive_cue_count"] = two_star_positive["tokens"].apply(lambda toks: count_terms(toks, positive_cue_terms))
two_star_positive["negative_issue_count"] = two_star_positive["tokens"].apply(lambda toks: count_terms(toks, negative_issue_terms))
two_star_positive["has_contrast_language"] = two_star_positive["review_text"].str.lower().str.contains(r"\b(but|however|although|not a deal breaker)\b", regex=True)

cue_summary = pd.DataFrame([
    {"Measure": "Reviews with contrast language", "Value": int(two_star_positive["has_contrast_language"].sum()), "Percent": f"{two_star_positive['has_contrast_language'].mean()*100:.1f}%"},
    {"Measure": "Avg positive cue term count", "Value": round(two_star_positive["positive_cue_count"].mean(), 2), "Percent": ""},
    {"Measure": "Avg negative issue term count", "Value": round(two_star_positive["negative_issue_count"].mean(), 2), "Percent": ""},
])
display(cue_summary)

# City split: are these mismatches concentrated geographically?
city_two_star_positive = (
    two_star_positive.groupby("city")
    .agg(
        mismatch_count=("review_id", "count"),
        avg_sentiment=("sentiment_score", "mean"),
        avg_positive_cues=("positive_cue_count", "mean"),
        avg_negative_issue_terms=("negative_issue_count", "mean"),
    )
    .reset_index()
)
city_two_star_base = two_star_all.groupby("city").size().rename("all_2star_count").reset_index()
city_total = df.groupby("city").size().rename("all_reviews").reset_index()
city_two_star_positive = city_two_star_positive.merge(city_two_star_base, on="city", how="outer").merge(city_total, on="city", how="left").fillna(0)
city_two_star_positive["mismatch_%_of_2star"] = (city_two_star_positive["mismatch_count"] / city_two_star_positive["all_2star_count"].replace(0, np.nan) * 100).round(1)
city_two_star_positive["mismatch_%_of_city_reviews"] = (city_two_star_positive["mismatch_count"] / city_two_star_positive["all_reviews"] * 100).round(1)
city_two_star_positive = city_two_star_positive.sort_values(["mismatch_count", "mismatch_%_of_2star"], ascending=False)
display(city_two_star_positive.round(2))

# Model split: are these mismatches more common for particular vehicles?
model_two_star_positive = (
    two_star_positive.groupby("vehicle_model")
    .agg(
        mismatch_count=("review_id", "count"),
        avg_sentiment=("sentiment_score", "mean"),
        avg_positive_cues=("positive_cue_count", "mean"),
        avg_negative_issue_terms=("negative_issue_count", "mean"),
    )
    .reset_index()
)
model_two_star_base = two_star_all.groupby("vehicle_model").size().rename("all_2star_count").reset_index()
model_total = df.groupby("vehicle_model").size().rename("all_model_reviews").reset_index()
model_two_star_positive = model_two_star_positive.merge(model_two_star_base, on="vehicle_model", how="outer").merge(model_total, on="vehicle_model", how="left").fillna(0)
model_two_star_positive["mismatch_%_of_2star"] = (model_two_star_positive["mismatch_count"] / model_two_star_positive["all_2star_count"].replace(0, np.nan) * 100).round(1)
model_two_star_positive["mismatch_%_of_model_reviews"] = (model_two_star_positive["mismatch_count"] / model_two_star_positive["all_model_reviews"] * 100).round(1)
model_two_star_positive = model_two_star_positive.sort_values(["mismatch_count", "mismatch_%_of_2star"], ascending=False)
display(model_two_star_positive.round(2))

# Representative examples: high positive sentiment but low rating.
representative_two_star_positive = two_star_positive.sort_values("sentiment_score", ascending=False)[
    ["review_id", "city", "vehicle_model", "rating", "sentiment_score", "positive_cue_count", "negative_issue_count", "review_text"]
].head(12)
display(representative_two_star_positive)

print("Interpretation:")
print("These are not simple sentiment errors. Many contain positive product language but still describe trust, charging, service, expectation or usability problems. The 2-star rating likely reflects the overall ownership judgement, while VADER reacts strongly to positive words inside mixed sentences.")

## 13. Product Experience vs Ownership Experience

In [ ]:
group_map = {
    "Product / vehicle experience": [
        "Driving performance",
        "Battery, range and charging",
        "Technology and software",
        "Build quality and interior",
        "Value and price",
    ],
    "Ownership / customer experience": [
        "Delivery and sales",
        "Service and repair",
        "Ownership trust and expectations",
        "Battery, range and charging",
        "Technology and software",
    ],
}

broad_rows = []
for group, themes in group_map.items():
    cols = [f"theme__{t}" for t in themes]
    mask = df[cols].any(axis=1)
    subset = df[mask]
    sent_dist = subset["sentiment_label"].value_counts(normalize=True).reindex(["Positive", "Neutral", "Negative"], fill_value=0) * 100
    broad_rows.append({
        "Broad Group": group,
        "Reviews mentioning group": int(mask.sum()),
        "Theme mentions": int(df[cols].sum().sum()),
        "Avg Rating": round(subset["rating"].mean(), 2),
        "Avg Sentiment": round(subset["sentiment_score"].mean(), 3),
        "Negative %": round(sent_dist["Negative"], 1),
        "Positive %": round(sent_dist["Positive"], 1),
    })
broad_summary = pd.DataFrame(broad_rows)
display(broad_summary)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(broad_summary["Broad Group"], broad_summary["Negative %"], color=["#3264a8", "#b54a4a"])
ax.set_title("Negative Sentiment: Product vs Ownership Experience")
ax.set_ylabel("Negative sentiment (%)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=10)
plt.tight_layout()
plt.savefig(FIG_DIR / "product_vs_ownership_negative_sentiment.png", dpi=180)
plt.show()

## 15. NMF Topic Modelling as a Comparison

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

vectorizer = TfidfVectorizer(min_df=4, max_df=0.65, ngram_range=(1, 2))
X = vectorizer.fit_transform(df["clean_text"])
terms = np.array(vectorizer.get_feature_names_out())

nmf_candidate_rows = []
nmf_models = {}
for k in [3, 4, 5, 6, 7, 8]:
    model = NMF(n_components=k, random_state=RANDOM_STATE, init="nndsvda", max_iter=800)
    W = model.fit_transform(X)
    nmf_models[k] = (model, W)
    nmf_candidate_rows.append({"topics": k, "reconstruction_error": model.reconstruction_err_})

nmf_candidates = pd.DataFrame(nmf_candidate_rows)
display(nmf_candidates.round(3))

selected_nmf_k = 5 if 5 in nmf_models else int(nmf_candidates.sort_values("reconstruction_error").iloc[0]["topics"])
nmf_model, W = nmf_models[selected_nmf_k]
print(f"Selected NMF topic count: {selected_nmf_k}")

nmf_terms = []
for topic_idx, topic_weights in enumerate(nmf_model.components_):
    top = terms[topic_weights.argsort()[::-1][:10]]
    nmf_terms.append({"topic": topic_idx, "top_terms": ", ".join(top)})
nmf_topic_terms = pd.DataFrame(nmf_terms)
display(nmf_topic_terms)

df["nmf_topic"] = W.argmax(axis=1)
df["nmf_topic_confidence"] = W.max(axis=1)
nmf_topic_summary = (
    df.groupby("nmf_topic")
    .agg(
        Review_Count=("review_id", "count"),
        Avg_Rating=("rating", "mean"),
        Avg_Sentiment=("sentiment_score", "mean"),
        Positive_Pct=("sentiment_label", lambda s: (s.eq("Positive").mean() * 100)),
        Negative_Pct=("sentiment_label", lambda s: (s.eq("Negative").mean() * 100)),
    )
    .reset_index()
    .round(2)
)
nmf_topic_summary["Review_Pct"] = (nmf_topic_summary["Review_Count"] / len(df) * 100).round(2)
nmf_topic_summary = nmf_topic_summary.merge(nmf_topic_terms, left_on="nmf_topic", right_on="topic").drop(columns="topic")
display(nmf_topic_summary)

## 16. Cross-Method Insight and Positive vs Negative Reviews

In [ ]:
high = df[df["rating"] >= 4]
low = df[df["rating"] <= 2]

def top_terms_for_subset(subset, top=15):
    counts = Counter(tok for toks in subset["tokens"] for tok in toks)
    return pd.DataFrame(counts.most_common(top), columns=["term", "count"])

def theme_share_for_subset(subset):
    rows = []
    for theme in theme_keywords:
        rows.append({
            "Theme": theme,
            "% Reviews": round(subset[f"theme__{theme}"].mean() * 100, 1),
            "Mentions": int(subset[f"theme__{theme}"].sum()),
        })
    return pd.DataFrame(rows).sort_values("% Reviews", ascending=False)

print("High satisfaction reviews (4-5 stars):", len(high))
display(top_terms_for_subset(high))
display(top_ngrams(high["tokens"], 2, 15))
display(theme_share_for_subset(high).head(8))

print("Low satisfaction reviews (1-2 stars):", len(low))
display(top_terms_for_subset(low))
display(top_ngrams(low["tokens"], 2, 15))
display(theme_share_for_subset(low).head(8))

## 17. Vehicle Model Comparison

In [ ]:
def top_themes_for_group(subset, top=3):
    shares = theme_share_for_subset(subset).head(top)
    return "; ".join([f"{row.Theme} ({row.Mentions})" for row in shares.itertuples()])

model_rows = []
for model, subset in df.groupby("vehicle_model"):
    model_rows.append({
        "Vehicle Model": model,
        "Reviews": len(subset),
        "Avg Rating": round(subset["rating"].mean(), 2),
        "Avg Sentiment": round(subset["sentiment_score"].mean(), 3),
        "Positive %": round(subset["sentiment_label"].eq("Positive").mean() * 100, 1),
        "Negative %": round(subset["sentiment_label"].eq("Negative").mean() * 100, 1),
        "Most mentioned themes": top_themes_for_group(subset),
    })
model_summary = pd.DataFrame(model_rows).sort_values("Reviews", ascending=False)
display(model_summary)

## 18. Location Analysis

In [ ]:
city_rows = []
for city, subset in df.groupby("city"):
    city_rows.append({
        "City": city,
        "Reviews": len(subset),
        "Avg Rating": round(subset["rating"].mean(), 2),
        "Avg Sentiment": round(subset["sentiment_score"].mean(), 3),
        "Negative %": round(subset["sentiment_label"].eq("Negative").mean() * 100, 1),
        "Major themes": top_themes_for_group(subset),
    })
city_summary = pd.DataFrame(city_rows).sort_values("Reviews", ascending=False)
display(city_summary)
